# 02 · Training and evaluation

Three Sentinel-1-only segmentation models of increasing capacity, trained on the global patches from
notebook 01 (8 classes, 2023–24) and tested on 9 unseen regions in **2025–26**.

| # | Model | Idea | Question it answers |
|---|---|---|---|
| 1 | **Baseline CNN** | 4 conv layers at full resolution, sees only a 90 m neighbourhood | How far do backscatter and local texture alone get us? |
| 2 | **U-Net** | Encoder-decoder with skip connections, trained from scratch | Does wider spatial context (> 2 km) plus sharp boundaries help? |
| 3 | **DeepLabV3+** | ImageNet-pretrained ResNet50 + atrous spatial pyramid pooling | Do pretrained features and explicit multi-scale context help further? |

### Avoiding over- and underfitting

| Measure | Against |
|---|---|
| Train / validation split **by region** (validation in other places than training) | optimistic scores |
| Augmentation: flips, rotations, brightness change, speckle noise | overfitting |
| Weight decay (AdamW), dropout in U-Net and DeepLabV3+ | overfitting |
| **Early stopping**: training stops by itself when validation mIoU has not improved for 8 epochs; the best epoch is kept | overfitting, wasted time |
| **Learning-rate reduction**: the learning rate halves when validation mIoU stalls for 3 epochs | underfitting (stuck training) |
| mIoU on training *and* validation data after every epoch, with an automatic verdict | detecting both |

Backpropagation and the weight updates are done by Keras (`model.fit`); the notebook only defines the models,
loss, data and stopping rules.

**Shared server:** the first cell picks the GPU with the most free memory. Set `SMOKE_TEST = True` first to
check the whole notebook in a few minutes.

In [ ]:
import os, shutil, subprocess
if shutil.which("nvidia-smi"):
    free = subprocess.run(["nvidia-smi", "--query-gpu=memory.free", "--format=csv,noheader,nounits"],
                          capture_output=True, text=True).stdout.split()
    os.environ["CUDA_VISIBLE_DEVICES"] = str(max(range(len(free)), key=lambda i: int(free[i])))
    print("Free GPU memory (MB):", free, "-> using GPU", os.environ["CUDA_VISIBLE_DEVICES"])
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import sys, json, time
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import tensorflow as tf
from tensorflow import keras

for gpu in tf.config.list_physical_devices("GPU"):
    tf.config.experimental.set_memory_growth(gpu, True)
keras.mixed_precision.set_global_policy("mixed_float16")
keras.utils.set_random_seed(42)

from src import config as C, model as M, evaluation as E, patching as P

CMAP = ListedColormap(C.CLASS_COLORS)
AUTOTUNE = tf.data.AUTOTUNE
print("TensorFlow", tf.__version__, "| GPUs:", tf.config.list_physical_devices("GPU"))

In [ ]:
SMOKE_TEST = False     # True: 64 random patches per split, 2 epochs - only checks that the code runs, scores are meaningless

BATCH = 16             # lower to 8 if the GPU runs out of memory
MAX_EPOCHS = 2 if SMOKE_TEST else 50
PATIENCE = 8           # early stopping: epochs without validation-mIoU improvement
LR_PATIENCE = 3        # halve the learning rate after this many epochs without improvement
LEARNING_RATES = {"baseline_cnn": 1e-3, "unet": 1e-3, "deeplabv3plus": 3e-4}   # lower for the pretrained encoder

REUSE_TRAINED = True   # a model whose final weights and log already exist is loaded instead of retrained (e.g. after a crash)

WEIGHTS_DIR = Path(os.environ.get("LULC_WEIGHTS_DIR", ROOT / "weights"))
if SMOKE_TEST:
    WEIGHTS_DIR = WEIGHTS_DIR / "smoke_test"     # never overwrite real results
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
stats = M.load_stats()
stats

## 1. Data

Training and validation patches are loaded into memory; the test sets are read from disk when evaluated.

In [ ]:
def load_split(split, in_memory=True):
    X = np.load(C.PATCH_DIR / f"X_{split}.npy", mmap_mode=None if in_memory else "r")
    y = np.load(C.PATCH_DIR / f"y_{split}.npy", mmap_mode=None if in_memory else "r")
    meta = pd.read_csv(C.PATCH_DIR / f"meta_{split}.csv")
    if SMOKE_TEST:   # 64 random patches, so that all regions can appear
        idx = np.sort(np.random.default_rng(0).choice(len(X), min(64, len(X)), replace=False))
        return np.asarray(X[idx]), np.asarray(y[idx]), meta.iloc[idx].reset_index(drop=True)
    return X, y, meta

X_train, y_train, meta_train = load_split("train")
X_val, y_val, meta_val = load_split("val")
pd.DataFrame({s: {"patches": len(m), "regions": m.aoi.nunique(), "months": m.month.nunique()}
              for s, m in [("train", meta_train), ("val", meta_val)]}).T

### Input pipeline

**Normalisation:** each channel is clipped to a fixed dB range and z-scored with the training-split mean and
standard deviation. **Augmentation** (training only):

| Augmentation | How | Why |
|---|---|---|
| Flips, 90° rotations | random horizontal flip, k × 90° rotation | land cover has no preferred orientation |
| Brightness change | the same random offset of ±1 dB on VV and VH (VV−VH ratio unchanged) | calibration and incidence-angle differences between scenes, orbits and satellites (Sentinel-1A in training, 1A + 1C in testing) |
| Speckle noise | Gaussian noise, σ = 0.3 dB | residual speckle depends on how many scenes a composite has |

A brightness change in dB is a multiplicative change of backscatter power, which is how such differences occur.
Large or per-channel changes are avoided because they would alter the physical meaning (e.g. make water look
like vegetation).

In [ ]:
BRIGHTNESS_DB = 1.0
NOISE_DB = 0.3


def augment_intensity(x, t):
    # brightness change and speckle-like noise, in dB, before normalisation
    x = tf.cast(x, tf.float32)
    offset = tf.random.uniform([], -BRIGHTNESS_DB, BRIGHTNESS_DB)
    x = x + tf.stack([offset, offset, 0.0])
    x = x + tf.random.normal(tf.shape(x), stddev=NOISE_DB)
    return x, t


def augment_geometry(x, t):
    k = tf.random.uniform([], 0, 4, tf.int32)
    x, t = tf.image.rot90(x, k), tf.image.rot90(t, k)
    flip = tf.random.uniform([]) < 0.5
    x = tf.cond(flip, lambda: tf.image.flip_left_right(x), lambda: x)
    t = tf.cond(flip, lambda: tf.image.flip_left_right(t), lambda: t)
    return x, t


def make_ds(X, y, training=False, index=None):
    index = np.arange(len(X)) if index is None else np.asarray(index)

    def gen():
        for i in (np.random.permutation(index) if training else index):
            yield X[i], y[i][..., None]

    ds = tf.data.Dataset.from_generator(gen, output_signature=(
        tf.TensorSpec((C.PATCH_PX, C.PATCH_PX, 3), tf.float16),
        tf.TensorSpec((C.PATCH_PX, C.PATCH_PX, 1), tf.uint8)))
    ds = ds.apply(tf.data.experimental.assert_cardinality(len(index)))   # lets Keras know the epoch length
    if training:
        ds = ds.map(augment_intensity, num_parallel_calls=AUTOTUNE)
    ds = ds.map(lambda x, t: (M.normalise(x, stats), tf.cast(t, tf.int32)), num_parallel_calls=AUTOTUNE)
    if training:
        ds = ds.map(augment_geometry, num_parallel_calls=AUTOTUNE)
    return ds.batch(BATCH).prefetch(AUTOTUNE)


train_ds = make_ds(X_train, y_train, training=True)
val_ds = make_ds(X_val, y_val)
# fixed subset of training patches (no augmentation) to measure training mIoU after each epoch
train_eval_idx = np.random.default_rng(0).choice(len(X_train), min(1000, len(X_train)), replace=False)
train_eval_ds = make_ds(X_train, y_train, index=train_eval_idx)

x, t = next(iter(train_ds))
print("batch:", x.shape, t.shape, "| channel means after normalisation:", np.round(x.numpy().mean((0, 1, 2)), 2))

## 2. Class imbalance and loss

Some classes (snow/ice, flooded vegetation) cover far fewer pixels than built-up or crops. The loss is
**class-weighted cross-entropy** (median-frequency weights, clipped to [0.2, 10]) plus **soft Dice**, which
measures overlap per class and is therefore less dominated by frequent classes. Pixels with unreliable labels
(index 255) are ignored.

In [ ]:
dist = P.class_distribution("train")
class_weights = M.median_frequency_weights(dist.pixels)
pd.DataFrame({"pixels_%": dist.percent.round(2).values, "loss_weight": class_weights.round(2)}, index=C.CLASSES)

## 3. Training procedure

The same procedure for all three models:

* AdamW (weight decay 1e-4), starting learning rate per model
* after every epoch: exact mIoU and overall accuracy on the **validation** set and on 1,000 fixed **training**
  patches
* learning rate × 0.5 after 3 epochs without validation-mIoU improvement
* stop after 8 epochs without improvement (at most 50); the weights of the best epoch are restored

In [ ]:
class EpochScores(keras.callbacks.Callback):
    # exact mIoU / OA from the full confusion matrix, added to the epoch logs for early stopping

    def __init__(self, val_ds, train_eval_ds):
        super().__init__()
        self.val_ds, self.train_eval_ds = val_ds, train_eval_ds

    def scores(self, ds):
        cm = np.zeros((M.NUM_CLASSES, M.NUM_CLASSES), np.int64)
        for x, t in ds:
            pred = np.argmax(self.model.predict_on_batch(x), -1)
            cm += E.confusion(t.numpy()[..., 0], pred)
        return E.scores(cm)

    def on_epoch_end(self, epoch, logs=None):
        v, tr = self.scores(self.val_ds), self.scores(self.train_eval_ds)
        logs.update(val_miou=v["mIoU"], val_oa=v["OA"], train_miou=tr["mIoU"], train_oa=tr["OA"])
        print(f"  mIoU train {tr['mIoU']:.3f} | val {v['mIoU']:.3f}   lr {float(self.model.optimizer.learning_rate):.1e}")


def diagnose(history):
    # compare training and validation mIoU at the best epoch
    best = history.val_miou.idxmax()
    tr, va = history.train_miou[best], history.val_miou[best]
    gap = tr - va
    if tr < 0.35:
        verdict = "underfitting: low mIoU even on training data - the model lacks capacity or context"
    elif gap > 0.15:
        verdict = "overfitting: clearly better on training than on validation regions"
    else:
        verdict = "good fit: training and validation mIoU are close"
    return dict(best_epoch=int(best), train_mIoU=round(float(tr), 4), val_mIoU=round(float(va), 4),
                gap=round(float(gap), 4), verdict=verdict)


def train(name):
    final, log, info_file = (WEIGHTS_DIR / f"{name}{s}" for s in [".keras", "_log.csv", "_info.json"])
    if REUSE_TRAINED and final.exists() and log.exists():
        model = keras.models.load_model(final, compile=False)
        history = pd.read_csv(log).set_index("epoch")
        history.index += 1
        minutes = json.loads(info_file.read_text())["minutes"] if info_file.exists() else float("nan")
        print(f"{name}: loaded from a previous run ({len(history)} epochs) - delete {final.name} to retrain")
        return model, history, dict(params=model.count_params(), epochs_run=len(history), minutes=minutes,
                                    **diagnose(history))

    model = M.MODELS[name]()
    # jit_compile=False: XLA rejects the float32 bilinear upsampling inside a float16 (mixed precision) graph
    model.compile(optimizer=keras.optimizers.AdamW(LEARNING_RATES[name], weight_decay=1e-4),
                  loss=M.make_loss(class_weights), jit_compile=False)
    ckpt = WEIGHTS_DIR / f"{name}_ckpt.keras"
    callbacks = [
        EpochScores(val_ds, train_eval_ds),
        keras.callbacks.ReduceLROnPlateau(monitor="val_miou", mode="max", factor=0.5, patience=LR_PATIENCE,
                                          min_lr=1e-6, verbose=1),
        keras.callbacks.EarlyStopping(monitor="val_miou", mode="max", patience=PATIENCE, min_delta=0.002,
                                      restore_best_weights=True, verbose=1),
        keras.callbacks.ModelCheckpoint(ckpt, monitor="val_miou", mode="max", save_best_only=True),
        keras.callbacks.CSVLogger(WEIGHTS_DIR / f"{name}_log.csv"),
    ]
    print(f"{name}: {model.count_params():,} parameters")
    t0 = time.time()
    hist = model.fit(train_ds, validation_data=val_ds, epochs=MAX_EPOCHS, callbacks=callbacks,
                     shuffle=False, verbose=2)   # shuffling happens in make_ds
    history = pd.DataFrame(hist.history)
    history.index += 1
    best = keras.models.load_model(ckpt, compile=False)
    best.save(final)                                    # weights only, no optimizer state
    info = dict(params=model.count_params(), epochs_run=len(history), minutes=round((time.time() - t0) / 60, 1),
                **diagnose(history))
    info_file.write_text(json.dumps({"minutes": info["minutes"]}))
    return best, history, info


def plot_history(history, title):
    fig, ax = plt.subplots(1, 3, figsize=(18, 4))
    ax[0].plot(history.loss, label="train"); ax[0].plot(history.val_loss, label="validation")
    ax[0].set_title("Loss (weighted CE + Dice)")
    ax[1].plot(history.train_miou, label="train"); ax[1].plot(history.val_miou, label="validation")
    ax[1].axvline(history.val_miou.idxmax(), color="grey", ls=":", label="best epoch")
    ax[1].set_title("mIoU")
    ax[2].plot(history.learning_rate); ax[2].set_yscale("log"); ax[2].set_title("Learning rate")
    for a in ax:
        a.set_xlabel("epoch"); a.grid(alpha=0.3)
    ax[0].legend(); ax[1].legend()
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


histories, infos, models = {}, {}, {}

## 4. Model 1 · Baseline CNN

Four 3 × 3 convolutions at full resolution, no pooling: each pixel is classified from a 9 × 9 pixel (90 m)
window. This tests how much the backscatter values and local texture alone can explain. Expected: decent on
classes with a distinctive signal (water: dark; built-up: bright double bounce), poor on classes that need
context (crops vs. rangeland, bare vs. snow).

In [ ]:
models["baseline_cnn"], histories["baseline_cnn"], infos["baseline_cnn"] = train("baseline_cnn")
plot_history(histories["baseline_cnn"], "Baseline CNN")
infos["baseline_cnn"]

## 5. Model 2 · U-Net

The encoder halves the resolution four times, so the deepest layer sees more than 2 km around each pixel (field
patterns, settlement structure, river networks). Skip connections pass the full-resolution features to the
decoder so boundaries stay sharp. Trained from scratch, so it makes no assumptions taken from optical images.

In [ ]:
models["unet"], histories["unet"], infos["unet"] = train("unet")
plot_history(histories["unet"], "U-Net")
infos["unet"]

## 6. Model 3 · DeepLabV3+

* **Encoder:** ResNet50 pretrained on ImageNet. SAR is not RGB, but low-level filters (edges, textures, blobs)
  transfer and are fine-tuned; this helps when labelled SAR data is limited.
* **ASPP:** parallel dilated convolutions (rates 6, 12, 18) plus a global branch look at each location at
  several scales at once, useful because the same class appears at very different sizes (a pond vs. a lake,
  a village vs. a megacity).
* **Decoder:** combines the deep features with 1/4-resolution features for sharper boundaries.

In [ ]:
models["deeplabv3plus"], histories["deeplabv3plus"], infos["deeplabv3plus"] = train("deeplabv3plus")
plot_history(histories["deeplabv3plus"], "DeepLabV3+")
infos["deeplabv3plus"]

## 7. Model comparison (training and validation)

In [ ]:
NAMES = {"baseline_cnn": "Baseline CNN", "unet": "U-Net", "deeplabv3plus": "DeepLabV3+"}
COLORS = {"baseline_cnn": "#6b7280", "unet": "#2563eb", "deeplabv3plus": "#dc2626"}

fig, ax = plt.subplots(1, 2, figsize=(16, 4.5))
for n, h in histories.items():
    ax[0].plot(h.loss, color=COLORS[n], label=f"{NAMES[n]} train")
    ax[0].plot(h.val_loss, color=COLORS[n], ls="--", label=f"{NAMES[n]} val")
    ax[1].plot(h.train_miou, color=COLORS[n], label=f"{NAMES[n]} train")
    ax[1].plot(h.val_miou, color=COLORS[n], ls="--", label=f"{NAMES[n]} val")
ax[0].set_title("Loss"); ax[1].set_title("mIoU (solid: training, dashed: validation)")
for a in ax:
    a.set_xlabel("epoch"); a.grid(alpha=0.3); a.legend(fontsize=8)
plt.tight_layout()
plt.savefig(WEIGHTS_DIR / "training_curves.png", dpi=150)
plt.show()

fit_table = pd.DataFrame({NAMES[n]: {"parameters (M)": i["params"] / 1e6, "epochs run": i["epochs_run"],
                                     "best epoch": i["best_epoch"], "train mIoU": i["train_mIoU"],
                                     "val mIoU": i["val_mIoU"], "gap": i["gap"], "minutes": i["minutes"],
                                     "verdict": i["verdict"]}
                          for n, i in infos.items()}).T
fit_table

## 8. Test evaluation · unseen regions, 2025–26

9 regions never used for training or model selection (Dubai, Mexico City, Dhaka, Kansas, Canterbury NZ, Gabon,
Pantanal, Atacama, Vatnajökull), all from the future period 2025–26.

In [ ]:
X_test, y_test, meta_test = load_split("test", in_memory=False)


def predict(model, X, batch=32):
    out = np.empty(X.shape[:3], np.uint8)
    for i in range(0, len(X), batch):
        out[i:i + batch] = np.argmax(model.predict_on_batch(M.normalise(np.asarray(X[i:i + batch]), stats)), -1)
    return out


results = {}
for n, model in models.items():
    pred = predict(model, X_test)
    cm = E.confusion(y_test, pred)
    results[n] = dict(pred=pred, cm=cm, **E.scores(cm))

test_table = pd.DataFrame({NAMES[n]: {"OA": r["OA"], "mIoU": r["mIoU"], "mean F1": r["mF1"]}
                           for n, r in results.items()}).T
test_table.round(4)

In [ ]:
support = next(iter(results.values()))["per_class"][["support_%"]]
per_class = pd.concat({"test pixels": support, **{NAMES[n]: r["per_class"][["IoU", "F1"]] for n, r in results.items()}},
                      axis=1)
per_class.round(3)

### Unseen regions vs. known regions

`test_temporal` applies the models to the *training* regions in 2025–26. The difference to the main test shows
how much error comes from new places rather than from the passage of time.

In [ ]:
temporal = {}
if (C.PATCH_DIR / "X_test_temporal.npy").exists():
    X_tt, y_tt, _ = load_split("test_temporal", in_memory=False)
    for n, model in models.items():
        s = E.scores(E.confusion(y_tt, predict(model, X_tt)))
        temporal[n] = {"OA": s["OA"], "mIoU": s["mIoU"], "mean F1": s["mF1"]}
    display(pd.concat({"test: unseen regions": test_table.astype(float),
                       "test_temporal: training regions": pd.DataFrame(temporal).T.rename(index=NAMES)},
                      axis=1).round(4))

## 9. Selected model

The model with the highest **validation** mIoU is selected (the test set plays no part in the choice) and
saved as `weights/best_model.keras` for the prototype.

In [ ]:
best_name = max(infos, key=lambda n: infos[n]["val_mIoU"])
print("Selected on validation mIoU:", NAMES[best_name])

cm = results[best_name]["cm"].astype(float)
with np.errstate(invalid="ignore"):
    cm_norm = cm / cm.sum(1, keepdims=True)
fig, ax = plt.subplots(figsize=(8.5, 7))
im = ax.imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(M.NUM_CLASSES), C.CLASSES, rotation=45, ha="right")
ax.set_yticks(range(M.NUM_CLASSES), C.CLASSES)
for i in range(M.NUM_CLASSES):
    for j in range(M.NUM_CLASSES):
        if not np.isnan(cm_norm[i, j]):
            ax.text(j, i, f"{cm_norm[i, j]:.2f}", ha="center", va="center", fontsize=8,
                    color="white" if cm_norm[i, j] > 0.5 else "black")
ax.set_xlabel("predicted"); ax.set_ylabel("reference (Dynamic World)")
ax.set_title(f"Test confusion matrix, row-normalised · {NAMES[best_name]}")
plt.colorbar(im, fraction=0.046)
plt.tight_layout()
plt.savefig(WEIGHTS_DIR / "confusion_matrix.png", dpi=150)
plt.show()

### Per region and per month

Per-region scores show how the model transfers to each unseen landscape; per-month scores show seasonal effects
(bare fields after harvest, flooding in the wet season, wet snow). The test regions span both hemispheres,
so calendar months mix opposite seasons.

In [ ]:
def grouped(y, pred, keys):
    rows = {}
    for k in sorted(set(keys)):
        m = np.asarray(keys) == k
        s = E.scores(E.confusion(y[m], pred[m]))
        rows[k] = {"OA": s["OA"], "mIoU": s["mIoU"], "patches": int(m.sum())}
    return pd.DataFrame(rows).T

pred_best = results[best_name]["pred"]
display(grouped(y_test, pred_best, meta_test.aoi.values).round(3))
by_month = grouped(y_test, pred_best, pd.to_datetime(meta_test.month).dt.month.values)
ax = by_month[["OA", "mIoU"]].plot(marker="o", figsize=(9, 3.5), title=f"Test scores by calendar month · {NAMES[best_name]}")
ax.set_xlabel("month"); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
idx = np.random.default_rng(1).choice(len(X_test), min(5, len(X_test)), replace=False)
fig, ax = plt.subplots(len(idx), 2 + len(results), figsize=(4 * (2 + len(results)), 4 * len(idx)), squeeze=False)
for row, i in enumerate(idx):
    ax[row, 0].imshow(np.asarray(X_test[i, ..., 0], np.float32), cmap="gray", vmin=-25, vmax=0)
    ax[row, 0].set_title(f"VV · {meta_test.aoi[i]} {meta_test.month[i]}", fontsize=9)
    ax[row, 1].imshow(np.ma.masked_equal(y_test[i], C.IGNORE), cmap=CMAP, vmin=0, vmax=M.NUM_CLASSES - 1,
                      interpolation="nearest")
    ax[row, 1].set_title("Dynamic World")
    for j, n in enumerate(results):
        ax[row, 2 + j].imshow(results[n]["pred"][i], cmap=CMAP, vmin=0, vmax=M.NUM_CLASSES - 1,
                              interpolation="nearest")
        ax[row, 2 + j].set_title(NAMES[n])
for a in ax.ravel():
    a.axis("off")
fig.legend([plt.Rectangle((0, 0), 1, 1, color=c) for c in C.CLASS_COLORS], C.CLASSES,
           loc="lower center", ncol=M.NUM_CLASSES)
plt.tight_layout(rect=(0, 0.03, 1, 1))
plt.show()

In [ ]:
models[best_name].save(WEIGHTS_DIR / "best_model.keras")

summary = {
    "selected_model": best_name,
    "classes": C.CLASSES,
    "training": {n: {k: (float(v) if isinstance(v, (int, float, np.floating)) else v) for k, v in i.items()}
                 for n, i in infos.items()},
    "test": {n: {"OA": float(r["OA"]), "mIoU": float(r["mIoU"]), "mF1": float(r["mF1"]),
                 "per_class_IoU": r["per_class"]["IoU"].round(4).to_dict(),
                 "per_class_F1": r["per_class"]["F1"].round(4).to_dict()} for n, r in results.items()},
    "test_temporal": {n: {k: float(v) for k, v in d.items()} for n, d in temporal.items()},
}
(WEIGHTS_DIR / "results.json").write_text(json.dumps(summary, indent=2))
print("Saved weights/best_model.keras (", NAMES[best_name], ") and weights/results.json")